# Chapter 3 — Classification
Companion notebook to `part1_fundamentals/03_classification.md`.
Uses MNIST via `fetch_openml` (downloads ~50MB the first time — needs internet).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.linear_model import SGDClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, cross_val_predict
from sklearn.metrics import (confusion_matrix, precision_score, recall_score, f1_score,
                              precision_recall_curve, roc_curve, roc_auc_score)

In [ ]:
mnist = fetch_openml('mnist_784', version=1, as_frame=False)
X, y = mnist["data"], mnist["target"]
X_train, X_test, y_train, y_test = X[:60000], X[60000:], y[:60000], y[60000:]

def plot_digit(data):
    plt.imshow(data.reshape(28, 28), cmap="binary")
    plt.axis("off")

plot_digit(X_train[0])
plt.title(f"Label: {y_train[0]}")
plt.show()

## Binary classifier: is it a 5?

In [ ]:
y_train_5 = (y_train == '5')
y_test_5 = (y_test == '5')

sgd_clf = SGDClassifier(random_state=42)
sgd_clf.fit(X_train, y_train_5)
print("Predict digit 0 is a 5?", sgd_clf.predict([X_train[0]]))

print("Cross-val accuracy:", cross_val_score(sgd_clf, X_train, y_train_5, cv=3, scoring="accuracy"))

## Confusion matrix, precision, recall, F1

In [ ]:
y_train_pred = cross_val_predict(sgd_clf, X_train, y_train_5, cv=3)
print(confusion_matrix(y_train_5, y_train_pred))
print("Precision:", precision_score(y_train_5, y_train_pred))
print("Recall:   ", recall_score(y_train_5, y_train_pred))
print("F1:       ", f1_score(y_train_5, y_train_pred))

## Precision/Recall trade-off & ROC curve

In [ ]:
y_scores = cross_val_predict(sgd_clf, X_train, y_train_5, cv=3, method="decision_function")
precisions, recalls, thresholds = precision_recall_curve(y_train_5, y_scores)

plt.plot(thresholds, precisions[:-1], "b--", label="Precision")
plt.plot(thresholds, recalls[:-1], "g-", label="Recall")
plt.xlabel("Threshold"); plt.legend(); plt.title("Precision/Recall vs threshold")
plt.show()

print("ROC AUC:", roc_auc_score(y_train_5, y_scores))

## Multiclass classification + error analysis

In [ ]:
forest_clf = RandomForestClassifier(random_state=42)
y_train_pred_multi = cross_val_predict(forest_clf, X_train, y_train, cv=3)
conf_mx = confusion_matrix(y_train, y_train_pred_multi)

plt.matshow(conf_mx, cmap="gray")
plt.title("Confusion matrix (10-class)")
plt.show()

print("Overall accuracy:", (y_train_pred_multi == y_train).mean())

## ✍️ Practice
1. Train a 3-detector. Find the threshold giving >=95% precision — what recall do you get?
2. Why should a spam filter prioritize precision while cancer screening prioritizes recall?
3. A model gets "99.1% accuracy" on 0.5%-fraud data — what should you ask for instead?
4. Which two digits get confused most in the confusion matrix above? Look at a few misclassified examples.

In [ ]:
# Your code / notes here